In [1]:
!pip install -U google-genai


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import json
import pandas as pd
import time
from google import genai
from google.genai import types

# 1. IL DIZIONARIO CON TUTTI I TUOI TEMI
PROMPTS = {
    "structural_conflict": """You are an expert media-content analyst.
Your task is to analyze the provided film SUBTITLES (dialogue only) and determine the salience of STRUCTURAL CONFLICT within the narrative.
Structural conflict refers to the degree to which the film’s narrative organization is driven by:
- persistent antagonism between opposing forces
- explicit threats, danger, or escalating tension
- sustained confrontation (physical, psychological, strategic, or institutional)
- high-stakes opposition shaping character decisions
- conflict functioning as the central engine of the story
IMPORTANT: You are NOT evaluating moral positioning, ideological stance, or justification of violence. You are ONLY measuring how central conflict is to the narrative structure.
SCORING SCALE (0–10)
0 — No meaningful conflict present.
1–2 — Minimal or background conflict; not structurally important.
3–4 — Conflict appears intermittently but does not drive the narrative.
5–6 — Conflict is relevant and recurring but shares centrality with other themes.
7–8 — Conflict is clearly central and shapes most major developments.
9–10 — Conflict is the dominant structural engine of the film; nearly all narrative progression is driven by sustained antagonism or threat.
ANALYTICAL RULES
- Base your analysis strictly on the provided subtitles.
- Do not infer unseen scenes or missing context.
- Do not assume genre conventions unless supported by dialogue evidence.
- Dialogue describing past events counts only if it significantly structures the ongoing narrative.
- Isolated arguments or brief disputes do NOT automatically imply high salience.
- Sustained escalation, recurring threat, and strategic confrontation increase salience.
- If the subtitles contain insufficient information to determine structural centrality, use the best evidence available but lower confidence accordingly.
OUTPUT FORMAT (return exactly this JSON and nothing else)
{
"theme_name": "structural_conflict",
"theme_salience": 0-10,
"justification": "3–6 sentences grounded explicitly in dialogue evidence explaining why conflict is or is not structurally central.",
"confidence": 0-1
}
Subtitles:
{{SUBTITLE_TEXT}}
""",

    "force_as_resolution": """You are an expert media-content analyst.
Your task is to analyze the provided film SUBTITLES (dialogue only) and determine the salience of FORCE/VIOLENCE AS RESOLUTION within the narrative.
Force/Violence as Resolution refers to the degree to which the film’s narrative suggests that physical force, violence, or aggressive coercion are the primary, inevitable, or most effective methods for solving problems:
- explicit dialogue endorsing or planning violence to resolve disputes
- non-violent alternatives being framed as weak, naive, or ineffective
- character progression being tied to their willingness or ability to use force
- conflicts and climaxes structurally demanding a violent or physically coercive resolution
- survival or justice depending strictly on the capacity to inflict harm
IMPORTANT: You are ONLY measuring how structurally central the application of force is to resolving the narrative's central issues.
SCORING SCALE (0–10)
0 — No meaningful reliance on force/violence to solve problems.
1–2 — Minimal or background mentions of force; problems are mostly solved otherwise.
3–4 — Force appears intermittently as a tool, but is not the primary problem-solving method.
5–6 — Force/violence is a relevant and recurring method for resolution, sharing centrality with other approaches.
7–8 — Force is clearly central; major narrative developments and obstacles require violent or aggressive solutions.
9–10 — Force/violence is the dominant engine of resolution.
ANALYTICAL RULES
- Base your analysis strictly on the provided subtitles.
- Do not infer unseen scenes or missing context.
- If the subtitles contain insufficient information, use the best evidence available but lower confidence accordingly.
OUTPUT FORMAT (return exactly this JSON and nothing else)
{
"theme_name": "force_as_resolution",
"theme_salience": 0-10,
"justification": "3–6 sentences grounded explicitly in dialogue evidence explaining why force/violence is or is not structurally central to problem-solving.",
"confidence": 0-1
}
Subtitles:
{{SUBTITLE_TEXT}}
""",

    "institutional_cynicism": """You are an expert media-content analyst.
Your task is to analyze the provided film SUBTITLES (dialogue only) and determine the salience of INSTITUTIONAL CYNICISM AND MISTRUST within the narrative.
Institutional cynicism refers to the degree to which the film’s narrative portrays formal institutions as:
- inherently corrupt, incompetent, or untrustworthy
- an obstacle to justice rather than a facilitator of it
- systems that characters must bypass, manipulate, or fight against to succeed
- fundamentally broken, requiring vigilantism or rogue actions
IMPORTANT: You are ONLY measuring how structurally central the distrust or failure of institutions is to the story.
SCORING SCALE (0–10)
0 — No meaningful institutional cynicism; institutions are helpful, absent, or neutral.
1–2 — Minimal cynicism; isolated complaints about rules or minor bureaucratic hurdles.
3–4 — Institutions are occasionally shown as flawed or unhelpful, but it doesn't drive the plot.
5–6 — Mistrust in institutions is a relevant and recurring theme that affects major character choices.
7–8 — Institutional failure/corruption is clearly central; the protagonist actively fights or circumvents the system.
9–10 — Cynicism is the dominant structural engine.
ANALYTICAL RULES
- Base your analysis strictly on the provided subtitles.
- Do not infer unseen scenes or missing context.
- If the subtitles contain insufficient information, use the best evidence available but lower confidence accordingly.
OUTPUT FORMAT (return exactly this JSON and nothing else)
{
"theme_name": "institutional_cynicism",
"theme_salience": 0-10,
"justification": "3–6 sentences grounded explicitly in dialogue evidence explaining why institutional cynicism is or is not structurally central.",
"confidence": 0-1
}
Subtitles:
{{SUBTITLE_TEXT}}
""",

    "power_dynamics": """You are an expert media-content analyst.
Your task is to analyze the provided film SUBTITLES (dialogue only) and determine the salience of POWER DYNAMICS AND DOMINATION within the narrative.
Power dynamics refers to the degree to which the film’s narrative is driven by struggles for control, authority, and hierarchical supremacy:
- dialogue focused on who gives orders and who obeys (domination vs. submission)
- strict enforcement or subversion of social, economic, or traditional hierarchies
- characters constantly assessing threats to their status or asserting dominance over others
- interpersonal relationships framed primarily as struggles for leverage or control
IMPORTANT: You are ONLY measuring how structurally central the struggle for dominance and hierarchy is to the narrative.
SCORING SCALE (0–10)
0 — No meaningful focus on power dynamics; relationships are egalitarian or cooperative.
1–2 — Minimal focus; power differences exist as background context.
3–4 — Power struggles appear intermittently but are secondary to other narrative goals.
5–6 — Power and domination are relevant and recurring themes that shape key relationships.
7–8 — Hierarchical struggles are clearly central.
9–10 — Power dynamics are the dominant structural engine; virtually all dialogue is framed around domination and status.
ANALYTICAL RULES
- Base your analysis strictly on the provided subtitles.
- Do not infer unseen scenes or missing context.
- If the subtitles contain insufficient information, use the best evidence available but lower confidence accordingly.
OUTPUT FORMAT (return exactly this JSON and nothing else)
{
"theme_name": "power_dynamics",
"theme_salience": 0-10,
"justification": "3–6 sentences grounded explicitly in dialogue evidence explaining why power dynamics are or are not structurally central.",
"confidence": 0-1
}
Subtitles:
{{SUBTITLE_TEXT}}
""",

    "extreme_individualism": """You are an expert media-content analyst.
Your task is to analyze the provided film SUBTITLES (dialogue only) and determine the salience of EXTREME INDIVIDUALISM within the narrative.
Extreme individualism refers to the degree to which the film’s narrative promotes the idea that individual action, self-reliance, and personal rules override collective cooperation:
- the "lone wolf" protagonist who works best alone and rejects teamwork
- dialogue emphasizing self-interest, personal survival, or individual glory over community needs
- groups, committees, or collective efforts being framed as slow, weak, or restrictive
- the ultimate resolution depending entirely on one person's unique will or capability breaking away from the group
IMPORTANT: You are ONLY measuring how structurally central the friction between the individual and the collective is to the narrative.
SCORING SCALE (0–10)
0 — No meaningful individualism; the narrative is highly collaborative.
1–2 — Minimal focus; the protagonist is capable but generally cooperates with others.
3–4 — Individualism appears intermittently, but teamwork is still valued.
5–6 — The tension between acting alone vs. acting together is a recurring theme.
7–8 — Extreme individualism is clearly central; the protagonist actively rejects help and solves problems alone.
9–10 — Individualism is the dominant structural engine.
ANALYTICAL RULES
- Base your analysis strictly on the provided subtitles.
- Do not infer unseen scenes or missing context.
- If the subtitles contain insufficient information, use the best evidence available but lower confidence accordingly.
OUTPUT FORMAT (return exactly this JSON and nothing else)
{
"theme_name": "extreme_individualism",
"theme_salience": 0-10,
"justification": "3–6 sentences grounded explicitly in dialogue evidence explaining why individualism is or is not structurally central.",
"confidence": 0-1
}
Subtitles:
{{SUBTITLE_TEXT}}
""",

    "perceived_threat": """You are an expert media-content analyst.
Your task is to analyze the provided film SUBTITLES (dialogue only) and determine the salience of PERCEIVED THREAT AND VULNERABILITY within the narrative.
Perceived threat refers to the degree to which the film’s narrative cultivates a "Mean World Syndrome," portraying the world as inherently dangerous and people as untrustworthy:
- dialogue emphasizing fear, paranoia, or the constant need for vigilance
- framing characters as highly vulnerable to sudden harm, betrayal, or crime
- a pervasive atmosphere of danger lurking just outside the characters' safe zones
- the assertion that strangers are dangerous and the outside world cannot be trusted
IMPORTANT: You are ONLY measuring how structurally central the feeling of fear, vulnerability, and environmental threat is to the narrative.
SCORING SCALE (0–10)
0 — No meaningful sense of threat.
1–2 — Minimal threat; isolated moments of fear that are quickly resolved.
3–4 — Vulnerability and threat appear intermittently, but characters generally feel safe.
5–6 — A sense of danger and need for vigilance is a recurring theme.
7–8 — Perceived threat is clearly central; fear of the outside world drives major decisions.
9–10 — Pervasive threat is the dominant structural engine; the narrative is built on paranoia.
ANALYTICAL RULES
- Base your analysis strictly on the provided subtitles.
- Do not infer unseen scenes or missing context.
- If the subtitles contain insufficient information, use the best evidence available but lower confidence accordingly.
OUTPUT FORMAT (return exactly this JSON and nothing else)
{
"theme_name": "perceived_threat",
"theme_salience": 0-10,
"justification": "3–6 sentences grounded explicitly in dialogue evidence explaining why perceived threat is or is not structurally central.",
"confidence": 0-1
}
Subtitles:
{{SUBTITLE_TEXT}}
"""
}

# 2. LA FUNZIONE DI CHIAMATA A GEMINI DEFINITIVA
def analizza_salienza_tema(tema, testo_sottotitoli):
    # Usiamo il client standard di Google
    client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
    
    # IL MODELLO CHE NON SI FERMA MAI: Gratuito, Illimitato e Intelligente
    model = "gemini-2.0-flash"
    
    prompt_completo = PROMPTS[tema].replace("{{SUBTITLE_TEXT}}", testo_sottotitoli)
    
    config = types.GenerateContentConfig(
        response_mime_type="application/json",
        temperature=0.1 # Massima coerenza accademica
    )
    
    response = client.models.generate_content(
        model=model,
        contents=prompt_completo,
        config=config,
    )
    return response.text

print("✅ MOTORE PRONTO: Gemini 2.0 Flash (Limite Giornaliero: ILLIMITATO!)")

✅ MOTORE AGGIORNATO: Filtri di sicurezza rimossi per analisi accademica completa.


In [1]:
import os
import json
import pandas as pd
import time

# ---------------------------------------------------------
# IMPOSTAZIONI: Cambia qui il tema che vuoi analizzare oggi!
# Temi disponibili: "structural_conflict", "force_as_resolution", 
# "institutional_cynicism", "power_dynamics", "extreme_individualism", "perceived_threat"
# ---------------------------------------------------------
THEME_TO_TEST = "institutional_cynicism"

# Definiamo dove salvare i dati di QUESTO tema
percorso_salvataggio = f"llm_outputs/salienza_{THEME_TO_TEST}.csv"

# 1. Carichiamo i metadati puliti
df = pd.read_csv("/work/data/metadata/metadata_final.csv", usecols=["imdb_id", "title"])
df = df.drop_duplicates(subset=["imdb_id"], keep="first")
print(f"🎬 Totale film nel catalogo: {len(df)}")

# 2. CONTROLLO SALVATAGGIO PROGRESSIVO (Checkpoint)
film_gia_processati = []
if os.path.exists(percorso_salvataggio):
    df_esistente = pd.read_csv(percorso_salvataggio)
    # Assicuriamoci che gli ID siano letti come stringhe per il confronto
    film_gia_processati = df_esistente['imdb_id'].astype(str).tolist()
    print(f"📂 Trovato file di salvataggio. Film già analizzati: {len(film_gia_processati)}")
else:
    # Creiamo un CSV vuoto con l'intestazione corretta
    colonne = ["imdb_id", "title", "theme_name", "theme_salience", "confidence", "justification"]
    pd.DataFrame(columns=colonne).to_csv(percorso_salvataggio, index=False)
    print("✨ Creato nuovo file di salvataggio.")

# Filtriamo i film rimanenti (forziamo il confronto tra stringhe)
df_da_fare = df[~df["imdb_id"].astype(str).isin(film_gia_processati)]
print(f"🚀 Film rimanenti da processare: {len(df_da_fare)}\n")
print("-" * 50)

# 3. IL CICLO DI ANALISI TURBO
for index, row in df_da_fare.iterrows():
    imdb_id = str(row["imdb_id"])
    titolo = str(row["title"])
    percorso_file = f"subtitles_clean/{imdb_id}.txt"
    
    print(f"Analizzando: {titolo} ({imdb_id})...")
    
    try:
        # Leggiamo i sottotitoli
        with open(percorso_file, "r", encoding="utf-8") as file:
            testo_sottotitoli = file.read()
            
        # Chiamata a Gemini (Assicurati che la Cella 1 usi "gemini-2.0-flash")
        risposta_testo = analizza_salienza_tema(THEME_TO_TEST, testo_sottotitoli)
        dati_json = json.loads(risposta_testo)
        
        # Strutturiamo la riga per il CSV
        nuova_riga = {
            "imdb_id": imdb_id,
            "title": titolo,
            "theme_name": dati_json.get("theme_name", THEME_TO_TEST),
            "theme_salience": dati_json.get("theme_salience", ""),
            "confidence": dati_json.get("confidence", ""),
            "justification": dati_json.get("justification", "")
        }
        
        # SALVATAGGIO IMMEDIATO
        pd.DataFrame([nuova_riga]).to_csv(percorso_salvataggio, mode='a', header=False, index=False)
        
        # PAUSA TURBO: Solo 1.5 secondi (Gemini 2 Flash regge 2000 richieste al minuto!)
        time.sleep(10)
        
    except FileNotFoundError:
        print(f"  ❌ File saltato: {percorso_file} non trovato.")
    except json.JSONDecodeError:
        print(f"  ⚠️ Errore JSON per {titolo}. Risposta non valida.")
        time.sleep(2)
    except Exception as e:
        errore = str(e)
        print(f"  🛑 Errore API per {titolo}: {errore}")
        # Se superiamo il limite (raro con Flash), pausa più lunga
        if "429" in errore or "quota" in errore.lower():
            print("  ⏳ Quota superata! Pausa forzata di 30 secondi...")
            time.sleep(30)
        else:
            time.sleep(5)

print(f"\n🎉 --- ANALISI DEL TEMA '{THEME_TO_TEST.upper()}' COMPLETATA! --- 🎉")

🎬 Totale film nel catalogo: 3788
📂 Trovato file di salvataggio. Film già analizzati: 3788
🚀 Film rimanenti da processare: 1

--------------------------------------------------
Analizzando: It: Chapter Two (tt7349950)...
  🛑 Errore API per It: Chapter Two: name 'analizza_salienza_tema' is not defined

🎉 --- ANALISI DEL TEMA 'INSTITUTIONAL_CYNICISM' COMPLETATA! --- 🎉


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=e08cfdf8-9b6e-44e8-b36c-3dd235d85ba1' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>